# AngioSim YOLOv8 Training - Drive Ready

This notebook keeps the dataset, augmented dataset, training runs, and best model in Google Drive so Colab runtime resets do not delete your work.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AngioSimAI")
DATASETS_DIR = PROJECT_DIR / "datasets"
MODELS_DIR = PROJECT_DIR / "models"
RUNS_DIR = PROJECT_DIR / "runs"

DATASET_ZIP = DATASETS_DIR / "angiosim-sutures.yolov8.zip"
RAW_DATASET_DIR = DATASETS_DIR / "dataset_raw"
AUG_DATASET_DIR = DATASETS_DIR / "dataset_aug"
MODEL_PATH = MODELS_DIR / "suture_yolov8_best.pt"
DATA_YAML = PROJECT_DIR / "data.yaml"

for path in [DATASETS_DIR, MODELS_DIR, RUNS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset zip:", DATASET_ZIP)
print("Model:", MODEL_PATH)


In [ ]:
!pip install ultralytics albumentations opencv-python -q

print("Ready")


## Upload dataset zip once

Run this cell only if `angiosim-sutures.yolov8.zip` is not already in Google Drive. After it is saved in Drive, you should not need to upload it again.


In [ ]:
from google.colab import files
import shutil

if DATASET_ZIP.exists():
    print("Dataset zip already exists in Drive:", DATASET_ZIP)
else:
    print("Upload your Roboflow YOLOv8 zip now")
    uploaded = files.upload()
    zip_names = [name for name in uploaded.keys() if name.endswith(".zip")]
    if not zip_names:
        raise FileNotFoundError("No .zip file uploaded")
    uploaded_zip = Path(zip_names[0])
    shutil.copy2(uploaded_zip, DATASET_ZIP)
    print("Saved zip to Drive:", DATASET_ZIP)


In [ ]:
import zipfile

if RAW_DATASET_DIR.exists() and any(RAW_DATASET_DIR.rglob("data.yaml")):
    print("Raw dataset already extracted:", RAW_DATASET_DIR)
else:
    if not DATASET_ZIP.exists():
        raise FileNotFoundError(f"Dataset zip not found: {DATASET_ZIP}")
    RAW_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        z.extractall(RAW_DATASET_DIR)
    print("Extracted raw dataset:", RAW_DATASET_DIR)

print("data.yaml files:")
for y in RAW_DATASET_DIR.rglob("data.yaml"):
    print("-", y)


In [ ]:
import yaml

yaml_files = list(RAW_DATASET_DIR.rglob("data.yaml"))
if not yaml_files:
    raise FileNotFoundError("Could not find Roboflow data.yaml")

RAW_DATA_YAML = yaml_files[0]
raw_cfg = yaml.safe_load(RAW_DATA_YAML.read_text())
class_names = raw_cfg.get("names", ["stitch"])
if isinstance(class_names, dict):
    class_names = [class_names[k] for k in sorted(class_names)]

print("Raw data.yaml:", RAW_DATA_YAML)
print("Classes:", class_names)


## Create augmented dataset only once

This reproduces your augmentation idea, but saves the result in Google Drive. If the augmented folder already exists, the cell skips work.


In [ ]:
import albumentations as A
import cv2
import os
import random
import shutil
from pathlib import Path

AUG_PER_IMAGE = 20  # change to 70 if you really need more copies

def find_split_dir(root, split, kind):
    matches = list(root.rglob(f"{split}/{kind}"))
    return matches[0] if matches else None

raw_train_images = find_split_dir(RAW_DATASET_DIR, "train", "images")
raw_train_labels = find_split_dir(RAW_DATASET_DIR, "train", "labels")
raw_valid_images = find_split_dir(RAW_DATASET_DIR, "valid", "images") or find_split_dir(RAW_DATASET_DIR, "val", "images")
raw_valid_labels = find_split_dir(RAW_DATASET_DIR, "valid", "labels") or find_split_dir(RAW_DATASET_DIR, "val", "labels")

if raw_train_images is None or raw_train_labels is None:
    raise FileNotFoundError("Could not find train/images and train/labels in Roboflow dataset")

out_train_images = AUG_DATASET_DIR / "train" / "images"
out_train_labels = AUG_DATASET_DIR / "train" / "labels"
out_valid_images = AUG_DATASET_DIR / "valid" / "images"
out_valid_labels = AUG_DATASET_DIR / "valid" / "labels"

if AUG_DATASET_DIR.exists() and any(out_train_images.glob("*.jpg")) and any(out_valid_images.glob("*.jpg")):
    print("Augmented dataset already exists:", AUG_DATASET_DIR)
else:
    for path in [out_train_images, out_train_labels, out_valid_images, out_valid_labels]:
        path.mkdir(parents=True, exist_ok=True)

    transform = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
        A.GaussNoise(p=0.4),
        A.Rotate(limit=15, p=0.5),
        A.RandomScale(scale_limit=0.1, p=0.4),
    ], bbox_params=A.BboxParams(format="yolo", label_fields=["labels"], min_visibility=0.2))

    image_paths = list(raw_train_images.glob("*.jpg")) + list(raw_train_images.glob("*.png")) + list(raw_train_images.glob("*.jpeg"))
    count = 0

    for img_path in image_paths:
        label_path = raw_train_labels / f"{img_path.stem}.txt"
        if not label_path.exists():
            continue

        shutil.copy2(img_path, out_train_images / img_path.name)
        shutil.copy2(label_path, out_train_labels / label_path.name)

        image = cv2.imread(str(img_path))
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        bboxes, labels = [], []
        for line in label_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                labels.append(int(parts[0]))
                bboxes.append([float(x) for x in parts[1:]])

        if not bboxes:
            continue

        for i in range(AUG_PER_IMAGE):
            try:
                aug = transform(image=image, bboxes=bboxes, labels=labels)
                if not aug["bboxes"]:
                    continue
                aug_img = cv2.cvtColor(aug["image"], cv2.COLOR_RGB2BGR)
                out_name = f"{img_path.stem}_aug{i:03d}"
                cv2.imwrite(str(out_train_images / f"{out_name}.jpg"), aug_img)
                with open(out_train_labels / f"{out_name}.txt", "w") as f:
                    for cls, box in zip(aug["labels"], aug["bboxes"]):
                        f.write(f"{int(cls)} " + " ".join(f"{x:.6f}" for x in box) + "\n")
                count += 1
            except Exception as e:
                pass

    if raw_valid_images and raw_valid_labels:
        valid_images = list(raw_valid_images.glob("*.jpg")) + list(raw_valid_images.glob("*.png")) + list(raw_valid_images.glob("*.jpeg"))
        for img_path in valid_images:
            label_path = raw_valid_labels / f"{img_path.stem}.txt"
            if label_path.exists():
                shutil.copy2(img_path, out_valid_images / img_path.name)
                shutil.copy2(label_path, out_valid_labels / label_path.name)
    else:
        all_train = list(out_train_images.glob("*.jpg"))
        random.shuffle(all_train)
        val_count = max(1, int(len(all_train) * 0.15))
        for img_path in all_train[:val_count]:
            label_path = out_train_labels / f"{img_path.stem}.txt"
            shutil.move(str(img_path), out_valid_images / img_path.name)
            if label_path.exists():
                shutil.move(str(label_path), out_valid_labels / label_path.name)

    print("Created augmented images:", count)

print("Train images:", len(list(out_train_images.glob("*.jpg"))))
print("Valid images:", len(list(out_valid_images.glob("*.jpg"))))


In [ ]:
import yaml

data_cfg = {
    "train": str(AUG_DATASET_DIR / "train" / "images"),
    "val": str(AUG_DATASET_DIR / "valid" / "images"),
    "nc": len(class_names),
    "names": class_names,
}

DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False, allow_unicode=True))
print(DATA_YAML.read_text())


## Train only if model does not already exist

If `suture_yolov8_best.pt` already exists in Drive, this cell will skip training.


In [ ]:
from ultralytics import YOLO

if MODEL_PATH.exists():
    print("Model already exists, skipping training:", MODEL_PATH)
else:
    model = YOLO("yolov8n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=50,
        imgsz=512,
        batch=8,
        patience=10,
        project=str(RUNS_DIR),
        name="suture_v1",
        exist_ok=True,
    )

    trained_best = RUNS_DIR / "suture_v1" / "weights" / "best.pt"
    if not trained_best.exists():
        found = list(RUNS_DIR.rglob("best.pt"))
        if not found:
            raise FileNotFoundError("Training finished, but best.pt was not found")
        trained_best = found[-1]

    MODEL_PATH.write_bytes(trained_best.read_bytes())
    print("Saved model to Drive:", MODEL_PATH)


## Dataset check

This confirms how many images and labels are available before or after training.


In [ ]:
from pathlib import Path

def count_files(folder, patterns=("*.jpg", "*.jpeg", "*.png", "*.txt")):
    folder = Path(folder)
    return {pattern: len(list(folder.glob(pattern))) for pattern in patterns}

for split in ["train", "valid"]:
    images_dir = AUG_DATASET_DIR / split / "images"
    labels_dir = AUG_DATASET_DIR / split / "labels"
    print(f"{split.upper()} images:", count_files(images_dir, ("*.jpg", "*.jpeg", "*.png")))
    print(f"{split.upper()} labels:", count_files(labels_dir, ("*.txt",)))


## Prediction helper

This is the part that uses your trained YOLO model to detect stitches and build the `df_stitches` table.


In [ ]:
from ultralytics import YOLO
import pandas as pd
import cv2
from google.colab.patches import cv2_imshow

def predict_stitches(image_path, conf=0.25, show=True):
    image_path = Path(image_path)
    if not MODEL_PATH.exists():
        raise FileNotFoundError(f"Model not found: {MODEL_PATH}")
    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    model = YOLO(str(MODEL_PATH))
    results = model.predict(source=str(image_path), conf=conf, save=True)

    all_boxes = []
    for r in results:
        if show:
            plotted = r.plot()
            plotted = cv2.cvtColor(plotted, cv2.COLOR_RGB2BGR)
            cv2_imshow(plotted)

        for box in r.boxes:
            x_center, y_center, width, height = box.xywhn[0].tolist()
            all_boxes.append({
                "x_center": x_center,
                "y_center": y_center,
                "width": width,
                "height": height,
                "confidence": float(box.conf[0].item()),
                "class_id": int(box.cls[0].item()),
            })

    df_stitches = pd.DataFrame(all_boxes)
    print(f"Processed image: {image_path}")
    print(f"Detected stitches: {len(df_stitches)}")

    if not df_stitches.empty:
        display(df_stitches.head(20))

    return df_stitches, results


## Suture metric analysis

This keeps the important logic from your original notebook: stitch count, spacing, uniformity, gaps, and OSATS-like score.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

VESSEL_DIAMETER_MM = 5.0

def calibrate_by_vessel_span(df, img_h, img_w, vessel_diameter_mm=VESSEL_DIAMETER_MM):
    df = df.copy()
    df["cy_px"] = df["y_center"] * img_h
    df["cx_px"] = df["x_center"] * img_w
    y_span_px = df["cy_px"].max() - df["cy_px"].min()

    if y_span_px < 10:
        df["h_px"] = df["height"] * img_h
        y_span_px = df["h_px"].max()
        method = "bbox height"
    else:
        method = "Y-span stitches"

    px_per_mm = y_span_px / vessel_diameter_mm if vessel_diameter_mm else 1
    return px_per_mm, method

def analyze_suture_metrics(df_stitches, image_path, vessel_diameter_mm=VESSEL_DIAMETER_MM, show_plot=True):
    image_path = Path(image_path)
    if df_stitches.empty or len(df_stitches) < 2:
        print("Need at least 2 detected stitches")
        return None

    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    img_h, img_w = img.shape[:2]

    df = df_stitches.sort_values("x_center").reset_index(drop=True).copy()
    df["cx_px"] = df["x_center"] * img_w
    df["cy_px"] = df["y_center"] * img_h
    df["w_px"] = df["width"] * img_w
    df["h_px"] = df["height"] * img_h

    px_per_mm, cal_method = calibrate_by_vessel_span(df, img_h, img_w, vessel_diameter_mm)
    if px_per_mm <= 0:
        px_per_mm = 1

    df["cx_mm"] = df["cx_px"] / px_per_mm
    df["cy_mm"] = df["cy_px"] / px_per_mm
    df["w_mm"] = df["w_px"] / px_per_mm
    df["h_mm"] = df["h_px"] / px_per_mm

    n = len(df)
    dists_mm = []
    for i in range(n - 1):
        p1 = np.array([df.loc[i, "cx_px"], df.loc[i, "cy_px"]])
        p2 = np.array([df.loc[i + 1, "cx_px"], df.loc[i + 1, "cy_px"]])
        dists_mm.append(float(np.linalg.norm(p2 - p1) / px_per_mm))

    mean_dist_mm = float(np.mean(dists_mm))
    std_dist_mm = float(np.std(dists_mm))
    min_dist_mm = float(np.min(dists_mm))
    max_dist_mm = float(np.max(dists_mm))
    cv_dist = float(std_dist_mm / mean_dist_mm * 100) if mean_dist_mm > 0 else 0
    uniformity = max(0.0, 1 - std_dist_mm / mean_dist_mm) if mean_dist_mm > 0 else 0

    mean_w_mm = float(df["w_mm"].mean())
    mean_h_mm = float(df["h_mm"].mean())
    size_cv = float(df["w_mm"].std() / mean_w_mm * 100) if mean_w_mm > 0 else 0
    size_uniformity = max(0.0, 1 - size_cv / 100)

    mean_cy_mm = float(df["cy_mm"].mean())
    deviation = df["cy_mm"].apply(lambda y: abs(y - mean_cy_mm))
    mean_dev_mm = float(deviation.mean())
    max_dev_mm = float(deviation.max())
    line_score = max(0, 100 - (mean_dev_mm / vessel_diameter_mm) * 100)

    gap_threshold = mean_dist_mm + 2 * std_dist_mm
    gaps = [
        {"from_stitch": i + 1, "to_stitch": i + 2, "distance_mm": round(d, 2)}
        for i, d in enumerate(dists_mm)
        if d > gap_threshold
    ]

    count_score = 100 if 8 <= n <= 12 else max(0, 100 - abs(n - 10) * 7)
    uniformity_score = uniformity * 100
    size_score = size_uniformity * 100
    gap_penalty = len(gaps) * 8

    osats = max(0, round(
        count_score * 0.25 +
        uniformity_score * 0.40 +
        size_score * 0.20 +
        line_score * 0.15 -
        gap_penalty,
        1,
    ))

    grade = (
        "Excellent" if osats >= 80 else
        "Good" if osats >= 60 else
        "Satisfactory" if osats >= 40 else
        "Needs practice"
    )

    metrics = {
        "stitch_count": n,
        "osats_score_100": osats,
        "osats_score_5": round(osats / 20, 1),
        "grade": grade,
        "uniformity_percent": round(uniformity * 100),
        "mean_distance_mm": round(mean_dist_mm, 2),
        "min_distance_mm": round(min_dist_mm, 2),
        "max_distance_mm": round(max_dist_mm, 2),
        "std_distance_mm": round(std_dist_mm, 2),
        "cv_distance_percent": round(cv_dist, 1),
        "mean_width_mm": round(mean_w_mm, 2),
        "mean_height_mm": round(mean_h_mm, 2),
        "size_uniformity_percent": round(size_uniformity * 100),
        "mean_line_deviation_mm": round(mean_dev_mm, 2),
        "max_line_deviation_mm": round(max_dev_mm, 2),
        "gaps": gaps,
        "calibration_method": cal_method,
        "px_per_mm": round(float(px_per_mm), 2),
    }

    print("=" * 54)
    print("ANGIOSIM SUTURE ANALYSIS")
    print("=" * 54)
    print(f"Image: {img_w} x {img_h} px")
    print(f"Calibration: {cal_method}, {metrics['px_per_mm']} px/mm")
    print(f"Stitches: {n}")
    print(f"Mean distance: {metrics['mean_distance_mm']} mm")
    print(f"Min / max distance: {metrics['min_distance_mm']} / {metrics['max_distance_mm']} mm")
    print(f"Uniformity: {metrics['uniformity_percent']}%")
    print(f"Gaps: {len(gaps)}")
    print(f"OSATS: {metrics['osats_score_100']} / 100 ({metrics['osats_score_5']} / 5)")
    print(f"Grade: {grade}")
    print("=" * 54)

    display(df[["cx_px", "cy_px", "cx_mm", "cy_mm", "w_mm", "h_mm", "confidence"]])

    if show_plot:
        plt.figure(figsize=(8, 4))
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        plt.scatter(df["cx_px"], df["cy_px"], c="cyan", s=60)
        for i, row in df.iterrows():
            plt.text(row["cx_px"] + 5, row["cy_px"] + 5, str(i + 1), color="yellow", fontsize=12)
        plt.title(f"Detected stitches: {n} | OSATS: {metrics['osats_score_100']}/100")
        plt.axis("off")
        plt.show()

    return metrics, df

def make_frontend_payload(metrics):
    if metrics is None:
        return None
    return {
        "osats_score": metrics["osats_score_5"],
        "stitch_count": metrics["stitch_count"],
        "uniformity_percent": metrics["uniformity_percent"],
        "feedback": (
            "Spacing is stable. Keep the same rhythm."
            if metrics["uniformity_percent"] >= 85 else
            "Spacing is acceptable, but can be more even."
            if metrics["uniformity_percent"] >= 70 else
            "Spacing is uneven. Try to keep stitch intervals more consistent."
        ),
    }


## Run analysis on one validation image

This tests the full flow: model prediction -> stitch table -> metrics -> frontend-style JSON.


In [ ]:
import random

valid_images = list((AUG_DATASET_DIR / "valid" / "images").glob("*.jpg"))
if not valid_images:
    raise FileNotFoundError("No validation images found")

example_image_path = random.choice(valid_images)
df_stitches, prediction_results = predict_stitches(example_image_path, conf=0.25, show=True)
analysis_result = analyze_suture_metrics(df_stitches, example_image_path, vessel_diameter_mm=VESSEL_DIAMETER_MM)

if analysis_result:
    metrics, analyzed_df = analysis_result
    frontend_payload = make_frontend_payload(metrics)
    print("Frontend payload:")
    print(frontend_payload)


## Optional: analyze several validation photos

Use this to confirm how many photos were processed and summarize model behavior.


In [ ]:
def analyze_validation_sample(limit=10, conf=0.25):
    images = list((AUG_DATASET_DIR / "valid" / "images").glob("*.jpg"))[:limit]
    rows = []

    print(f"Images selected: {len(images)}")
    for image_path in images:
        df, _ = predict_stitches(image_path, conf=conf, show=False)
        result = analyze_suture_metrics(df, image_path, show_plot=False)
        if result is None:
            rows.append({"image": image_path.name, "stitch_count": 0, "osats_score_100": None, "uniformity_percent": None})
            continue
        metrics, _ = result
        rows.append({"image": image_path.name, **metrics})

    summary = pd.DataFrame(rows)
    print(f"Processed photos: {len(summary)}")
    display(summary[["image", "stitch_count", "osats_score_100", "osats_score_5", "uniformity_percent", "grade"]])
    return summary

# Uncomment when you want batch checking:
# validation_summary = analyze_validation_sample(limit=10, conf=0.25)
